# Phase 6 Lab — Reference Solution

**Phase:** Machine Learning Fundamentals  
**Scenario:** A team needs a leakage-safe baseline for churn prioritization under a fixed outreach capacity.

**Deliverable:** A reproducible model pipeline with baseline comparison, cross-validation, threshold policy, calibration check, and error slices.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Create a deployment-like split and preserve a final holdout.
2. Establish dummy and simple linear baselines.
3. Put imputation, encoding, and scaling inside a pipeline.
4. Cross-validate using metrics appropriate for imbalance.
5. Choose a threshold under an outreach-capacity or recall constraint.
6. Inspect calibration and subgroup error rates.
7. Document reproducibility metadata.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df=pd.read_csv(DATA_DIR/"customer_churn.csv")
X=df.drop(columns=["customer_id","churn"]); y=df["churn"]
num=X.select_dtypes(include="number").columns
cat=X.select_dtypes(exclude="number").columns
prep=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                     ("encode",OneHotEncoder(handle_unknown="ignore"))]),cat),
])
model=Pipeline([("prep",prep),("model",LogisticRegression(max_iter=1000,class_weight="balanced"))])
Xdev,Xtest,ydev,ytest=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
cv=StratifiedKFold(5,shuffle=True,random_state=42)
scores=cross_validate(model,Xdev,ydev,cv=cv,scoring=["roc_auc","average_precision"])
display(pd.DataFrame(scores)[["test_roc_auc","test_average_precision"]].describe())
model.fit(Xdev,ydev)
prob=model.predict_proba(Xtest)[:,1]
precision,recall,thresholds=precision_recall_curve(ytest,prob)
operating=pd.DataFrame({"threshold":np.r_[thresholds,1],"precision":precision,"recall":recall})
chosen=operating[operating.recall>=.80].sort_values("precision",ascending=False).iloc[0]
print("Holdout ROC-AUC:",roc_auc_score(ytest,prob))
print("Holdout PR-AUC:",average_precision_score(ytest,prob))
display(chosen.to_frame("chosen"))

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.